In [1]:
import pandas as pd
from pypdf import PdfReader
import os
import re
import json

In [2]:
pdf_path = "../data/TR_PDFs/TR#478 [Game Programming NC III].pdf"
#pdf_path = "../data/TR_PDFs/TR#2228 [Food Production (Professional Cookery) NC IV].pdf"
#pdf_path = "../data/TR_PDFs/TR#2232 [Front Office Services NC IV].pdf"

In [16]:
def get_units_of_competencies(pdf_path):
    """
        Given a Training Regulation PDF from TESDA, 
        return the list of competency standards of that specific TR
    """
    # Combine all PDF pages into one string
    reader = PdfReader(pdf_path)
    full_text = "\n".join(
        page.extract_text() or ""
        for page in reader.pages[:10] # we are only reading the first 10 pages
    )

    codes = []
    for line in full_text.splitlines():
        # Remove spaces/tabs, but NOT the newline separating rows
        match = re.match(
            r"(\w\s*){3}(\d\s*){6}\b",
            line
        )
        if match:
            codes.append(match.group().replace(" ", ""))
    return codes

In [21]:
# open the json that has the training_regulations and its data id
tr_competencies_filename = "../data/tr_competencies.json"

if not os.path.exists(tr_competencies_filename):
    with open('../data/tesda_training_regulations.json', 'r', encoding='utf-8') as file:
        training_regulations = json.load(file)
    # switch the key and value pair
    training_regulations = {
        data_id: training_regulation
        for training_regulation, data_id in training_regulations.items()
    }

    tr_competencies = {
        'Training Regulation': [],
        'Data ID': [],
        'Competency List': []
    }

    base_root = '../data/TR_PDFs'
    for file in sorted(os.listdir(base_root)):
        print(file)

        # get data id and name
        data_id = int(re.search(r'#(\d*).pdf', file).group(1))
        #print(data_id, file)
        training_regulation_name = training_regulations[data_id]

        # get the competencies
        filepath = os.path.join(base_root, file)
        competency_list = get_units_of_competencies(filepath)

        # append the bunch
        tr_competencies['Training Regulation'].append(training_regulation_name)
        tr_competencies['Data ID'].append(data_id)
        tr_competencies['Competency List'].append(competency_list)

        with open(tr_competencies_filename, "w") as file:
            json.dump(tr_competencies, file)

In [24]:
tr_competencies_df = pd.read_json(tr_competencies_filename)
tr_competencies_df

,Training Regulation,Data ID,Competency List
0,2D Animation NC III,123,"[500311109, 500311110, 500311111, 500311112, 5..."
1,2D Game Art Development NC III,464,"[500311109, 500311110, 500311111, 500311112, 5..."
2,3D Animation NC III,251,"[500311109, 500311110, 500311111, 500311112, 5..."
3,3D Game Art Development NC III,477,"[500311109, 500311110, 500311111, 500311112, 5..."
4,5-Axis CNC Machine Operation NC III,1901,"[400311319, 400311320, 400311321, 400311322, 4..."
...,...,...,...
318,Visual Graphic Design NC III,383,"[500311109, 500311110, 500311111, 500311112, 5..."
319,Warehousing Services NC III,2234,"[400311319, 400311320, 400311321, 400311322, 4..."
320,Warehousing Services NC II,1801,"[500311105, 500311106, 500311107, 500311108, L..."
321,Warehousing Services NC IV,2235,"[500311401, 500311402, 500311403, 500311404, 5..."
